imports y ruta al CSV

In [1]:


import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/twcs.csv"  # desde notebooks/
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head(5)


Shape: (2811774, 7)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


columnas y nulos

In [2]:
df.columns, df.isna().mean().sort_values(ascending=False).head(15)


(Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
        'response_tweet_id', 'in_response_to_tweet_id'],
       dtype='object'),
 response_tweet_id          0.370097
 in_response_to_tweet_id    0.282503
 tweet_id                   0.000000
 author_id                  0.000000
 inbound                    0.000000
 created_at                 0.000000
 text                       0.000000
 dtype: float64)

solo mensajes del cliente (inbound)

In [3]:
# inbound suele venir como True/False o como booleano
df["inbound"].value_counts(dropna=False)

df_in = df[df["inbound"] == True].copy()
print("Inbound shape:", df_in.shape)
df_in[["tweet_id", "author_id", "created_at", "text"]].head(10)


Inbound shape: (1537843, 7)


,tweet_id,author_id,created_at,text
1,2,115712,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that
2,3,115712,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...
4,5,115712,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.
6,8,115712,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service
8,12,115713,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...
10,16,115713,Tue Oct 31 20:00:43 +0000 2017,@sprintcare Since I signed up with you....Sinc...
12,18,115713,Tue Oct 31 19:56:01 +0000 2017,@115714 y’all lie about your “great” connectio...
14,20,115715,Tue Oct 31 22:03:34 +0000 2017,"@115714 whenever I contact customer support, t..."
16,22,115716,Tue Oct 31 22:16:48 +0000 2017,@Ask_Spectrum Would you like me to email you a...
18,26,115716,Tue Oct 31 22:19:56 +0000 2017,@Ask_Spectrum I received this from your corpor...


limpieza mínima (texto vacío / duplicados)

In [4]:
df_in["text"] = df_in["text"].astype(str).str.strip()

df_in = df_in[df_in["text"].str.len() > 0].copy()
df_in = df_in.drop_duplicates(subset=["tweet_id"]).copy()

print("After basic cleaning:", df_in.shape)


After basic cleaning: (1537843, 7)


longitud de textos (para saber qué tan “ticket” es)

In [5]:
df_in["n_chars"] = df_in["text"].str.len()
df_in["n_words"] = df_in["text"].str.split().str.len()

df_in[["n_chars", "n_words"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95])


,n_chars,n_words
count,1.537843e+06,1.537843e+06
mean,1.099361e+02,1.872803e+01
std,5.716449e+01,1.064469e+01
min,1.000000e+00,1.000000e+00
50%,1.090000e+02,1.800000e+01
75%,1.400000e+02,2.500000e+01
90%,1.730000e+02,3.100000e+01
95%,2.240000e+02,3.900000e+01
max,5.130000e+02,1.370000e+02


sample aleatorio (para “olfatear” el contenido)

In [6]:
pd.set_option("display.max_colwidth", 200)
df_in.sample(10, random_state=42)[["created_at", "text"]]


,created_at,text
26861,Wed Nov 01 14:03:55 +0000 2017,"@AppleSupport Basically for a chat to be opened from call log, the message app should be opened/running in background. Otherwise, it takes twice."
211386,Thu Oct 05 14:08:30 +0000 2017,"@AppleSupport iOS 11.02 and Watchos4.0: No icon for Twitter notifications. Restart of iphone/watch, notif. off/on does not help. What to do? https://t.co/Jd98V9OvIu"
78521,Thu Nov 30 17:14:45 +0000 2017,"Dear god not again,@AppleSupport https://t.co/5Zf0Mnd6SI"
1225222,Mon Oct 16 13:33:22 +0000 2017,"@ATVIAssist Hi there! If I buy Call of Duty WWII on steam today, do I have instant access to the open beta multiplayer?"
194583,Thu Oct 05 08:01:11 +0000 2017,Hi @Safaricom_Care why can't I pay my my Dstv texts says the Org is unavailable
1321593,Thu Nov 02 18:45:26 +0000 2017,trying to buy my kid nibling a keyboard for an upcoming birthday - why is the pink one more expensive than the blue one??? @116062 https://t.co/DuRHF6ntXV
835678,Fri Oct 13 22:23:23 +0000 2017,@GWRHelp https://t.co/EzzirrQuyS
685783,Thu Nov 23 00:11:25 +0000 2017,"@131749 recibi llamadas de celulares 3113552287 y 3114248440 de supuestos representantes suyos, avisando q me gané unos cupones hoteleros, solo tenía q revalidar la info de mi tarjeta. Ojo! Súper ..."
1731163,Mon Oct 23 07:53:21 +0000 2017,"@GloCare hi for 2days now I have not being able to make calls,text or even check my balance. As soon as I try making a call d line cuts.."
2338336,Wed Nov 15 21:21:24 +0000 2017,@AmazonHelp Are you guys having problems with the launch of Amazon Music in Canada? Keeps saying my session has expired. Already cleared cookies and temp files. Please Help! https://t.co/1r8mdH5ABB


quedarte con un subset manejable

In [7]:
# Recomendación: 50k–100k para iniciar
df_small = df_in.sample(80000, random_state=42).copy()
df_small.shape


(80000, 9)

guardar subset limpio (para el pipeline)

In [8]:
OUT_PATH = "../data/processed/tickets_inbound_clean.csv"
df_small.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)


Saved: ../data/processed/tickets_inbound_clean.csv
